# ReviewSense AI — Sentiment Model Training

This notebook reproduces the project's TF-IDF + Logistic Regression training and evaluation workflow. Run the cells from top to bottom. When prompted, upload `Dataset-SA.csv` from your project.

The test set is held aside until the final cell. Do not use test results to tune the model.

In [ ]:
# Cell 1 — Imports and upload the dataset
import re
import unicodedata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import files
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline

uploaded = files.upload()
csv_names = [name for name in uploaded if name.lower().endswith(".csv")]
if not csv_names:
    raise ValueError("Upload the Dataset-SA.csv file.")

raw_data = pd.read_csv(csv_names[0], low_memory=False)
print("Dataset shape:", raw_data.shape)
print("Columns:", raw_data.columns.tolist())

## 1. Clean and combine the text

We combine `Review` and `Summary` because either field can be missing. We remove only malformed rating rows, rows with no text, and unexpected labels. The rating and product details are not model inputs.

In [ ]:
# Cell 2 — Clean text and labels
EXPECTED_LABELS = ["negative", "neutral", "positive"]
TEXT_SEPARATOR = " [SEP] "

required_columns = {"Review", "Summary", "Sentiment", "Rate"}
missing_columns = required_columns - set(raw_data.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

def clean_text(value):
    """Normalize Unicode and whitespace while preserving sentiment wording."""
    if pd.isna(value):
        return ""
    value = unicodedata.normalize("NFKC", str(value))
    return re.sub(r"\s+", " ", value).strip()

def combine_review_text(review, summary):
    review_text = clean_text(review)
    summary_text = clean_text(summary)
    parts = [part for part in (review_text, summary_text) if part]
    return TEXT_SEPARATOR.join(parts)

data = raw_data.copy()
data["text"] = [
    combine_review_text(review, summary)
    for review, summary in zip(data["Review"], data["Summary"])
]
data["label"] = data["Sentiment"].astype("string").str.strip().str.casefold()

numeric_rate = pd.to_numeric(data["Rate"], errors="coerce")
malformed_rate = numeric_rate.isna() | ~numeric_rate.isin([1, 2, 3, 4, 5])
empty_text = data["text"].eq("")
unknown_label = ~data["label"].isin(EXPECTED_LABELS)
remove_row = malformed_rate | empty_text | unknown_label

print("Input rows:", len(data))
print("Malformed Rate rows removed:", int(malformed_rate.sum()))
print("Empty-text rows removed:", int((empty_text & ~malformed_rate).sum()))
print("Unexpected-label rows removed:", int((unknown_label & ~malformed_rate & ~empty_text).sum()))

data = data.loc[~remove_row, ["text", "label"]].reset_index(drop=True)
print("Usable rows:", len(data))
print("\nSentiment counts:")
print(data["label"].value_counts().reindex(EXPECTED_LABELS))

## 2. Make the untouched test split

We group by the exact combined text before splitting. That keeps identical model inputs on the same side of the split. Stratification aims to preserve the class balance.

In [ ]:
# Cell 3 — Create a leakage-safe train/test split
RANDOM_SEED = 42
LABELS = ["negative", "neutral", "positive"]

outer_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)
train_indices, test_indices = next(
    outer_splitter.split(data["text"], data["label"], groups=data["text"])
)

train_data = data.iloc[train_indices].reset_index(drop=True)
test_data = data.iloc[test_indices].reset_index(drop=True)

overlap = set(train_data["text"]).intersection(test_data["text"])
if overlap:
    raise RuntimeError(f"Leakage check failed: {len(overlap)} text groups overlap.")
if set(train_data["label"].unique()) != set(LABELS):
    raise RuntimeError("Training split is missing a sentiment class.")
if set(test_data["label"].unique()) != set(LABELS):
    raise RuntimeError("Test split is missing a sentiment class.")

print(f"Training rows: {len(train_data):,}")
print(f"Test rows: {len(test_data):,}")
print("Exact text groups shared between train and test:", len(overlap))
print("\nTraining class counts:")
print(train_data["label"].value_counts().reindex(LABELS))
print("\nTest class counts:")
print(test_data["label"].value_counts().reindex(LABELS))

## 3. Train the baseline and compare class weights

We create a second grouped split inside the training data for model selection. TF-IDF learns its vocabulary from the training fold only. The final test set remains unused.

In [ ]:
# Cell 4 — Train TF-IDF + Logistic Regression and compare class weights
inner_splitter = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_SEED,
)
fit_indices, validation_indices = next(
    inner_splitter.split(
        train_data["text"],
        train_data["label"],
        groups=train_data["text"],
    )
)

fit_data = train_data.iloc[fit_indices]
validation_data = train_data.iloc[validation_indices]

validation_overlap = set(fit_data["text"]).intersection(validation_data["text"])
if validation_overlap:
    raise RuntimeError("Identical text groups crossed into validation.")

vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents="unicode",
    ngram_range=(1, 2),
    min_df=2,
    max_features=300_000,
    sublinear_tf=True,
)
fit_features = vectorizer.fit_transform(fit_data["text"])
validation_features = vectorizer.transform(validation_data["text"])

def print_metrics(title, y_true, y_pred):
    print(f"\n{title}")
    print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
    print(
        "Macro F1:",
        f"{f1_score(y_true, y_pred, labels=LABELS, average='macro', zero_division=0):.4f}",
    )
    print(
        "Weighted F1:",
        f"{f1_score(y_true, y_pred, labels=LABELS, average='weighted', zero_division=0):.4f}",
    )
    print("\nPer-class precision, recall, and F1:")
    print(
        classification_report(
            y_true,
            y_pred,
            labels=LABELS,
            target_names=LABELS,
            zero_division=0,
        )
    )
    ConfusionMatrixDisplay.from_predictions(
        y_true,
        y_pred,
        labels=LABELS,
        display_labels=LABELS,
        cmap="Blues",
        colorbar=False,
    )
    plt.title(f"{title} — Confusion Matrix")
    plt.tight_layout()
    plt.show()

classifiers = {}
for weight_setting in ("balanced", None):
    classifier = LogisticRegression(
        class_weight=weight_setting,
        max_iter=1000,
        random_state=RANDOM_SEED,
    )
    classifier.fit(fit_features, fit_data["label"])
    validation_predictions = classifier.predict(validation_features)
    display_name = "balanced weights" if weight_setting == "balanced" else "no class weights"
    classifiers[weight_setting] = classifier
    print_metrics(display_name, validation_data["label"], validation_predictions)

print("TF-IDF vocabulary size:", len(vectorizer.vocabulary_))

## 4. Tune the neutral decision threshold

The validation comparison favored no class weights. This search changes only the neutral decision threshold and selects the value with the highest validation macro F1. A neutral F1 score breaks ties.

In [ ]:
# Cell 5 — Select the neutral threshold using validation data only
unweighted_model = classifiers[None]
validation_probabilities = unweighted_model.predict_proba(validation_features)
class_positions = {
    label: list(unweighted_model.classes_).index(label)
    for label in LABELS
}

p_negative = validation_probabilities[:, class_positions["negative"]]
p_neutral = validation_probabilities[:, class_positions["neutral"]]
p_positive = validation_probabilities[:, class_positions["positive"]]

def apply_neutral_threshold(probabilities, classes, threshold):
    positions = {label: list(classes).index(label) for label in LABELS}
    negative = probabilities[:, positions["negative"]]
    neutral = probabilities[:, positions["neutral"]]
    positive = probabilities[:, positions["positive"]]
    return np.where(
        neutral >= threshold,
        "neutral",
        np.where(negative >= positive, "negative", "positive"),
    )

best_threshold = None
best_predictions = None
best_key = (-1.0, -1.0)

for threshold in np.linspace(0.05, 0.95, 91):
    predictions = apply_neutral_threshold(
        validation_probabilities,
        unweighted_model.classes_,
        threshold,
    )
    macro_f1 = f1_score(
        validation_data["label"],
        predictions,
        labels=LABELS,
        average="macro",
        zero_division=0,
    )
    neutral_f1 = f1_score(
        validation_data["label"],
        predictions,
        labels=["neutral"],
        average="macro",
        zero_division=0,
    )
    if (macro_f1, neutral_f1) > best_key:
        best_key = (macro_f1, neutral_f1)
        best_threshold = float(threshold)
        best_predictions = predictions

print(f"Selected neutral threshold: {best_threshold:.2f}")
print_metrics(
    "Unweighted Logistic Regression with selected neutral threshold",
    validation_data["label"],
    best_predictions,
)

## 5. Final evaluation on the test set

After choosing the model and threshold using validation data, we refit on all training rows and evaluate the test set once. Use these results as the final generalization estimate; do not tune against the test results.

In [ ]:
# Cell 6 — Refit on all training rows, then evaluate the held-out test set
final_pipeline = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                lowercase=True,
                strip_accents="unicode",
                ngram_range=(1, 2),
                min_df=2,
                max_features=300_000,
                sublinear_tf=True,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=1000,
                random_state=RANDOM_SEED,
            ),
        ),
    ]
)

final_pipeline.fit(train_data["text"], train_data["label"])
test_probabilities = final_pipeline.predict_proba(test_data["text"])
final_classifier = final_pipeline.named_steps["classifier"]
test_predictions = apply_neutral_threshold(
    test_probabilities,
    final_classifier.classes_,
    best_threshold,
)

print_metrics(
    "Final holdout test results",
    test_data["label"],
    test_predictions,
)

## 6. Save the model for the web application

This saves the trained text pipeline and selected neutral threshold together. The file can be downloaded from Colab. The probability values are model estimates, not calibrated certainty.

In [ ]:
# Cell 7 — Export a model artifact
import joblib

artifact = {
    "pipeline": final_pipeline,
    "neutral_threshold": best_threshold,
    "labels": LABELS,
}
model_filename = "reviewsense_tfidf_logreg.joblib"
joblib.dump(artifact, model_filename)
print("Saved:", model_filename)
files.download(model_filename)